# 01 — Exploratory Data Analysis
**RO-Voting-Prediction** | NUTS 3 (județ) level | Eurostat 2022 baseline + INS Census 2021 + roaep.ro 2024–2025

### Notebook objectives
1. Audit data completeness before modelling
2. Profile electoral results (2024 parliamentary as primary target)
3. Map geographic distribution of key targets and features
4. Test bivariate relationships for each analytical hypothesis
5. Validate foundational structural variables (ethnic bloc, Georgescu→Simion)
6. Identify outliers and flag modelling concerns (VIF, leverage points)

> **Causal language note:** All associations reported here are observational.
> Phrases like *'predicts'*, *'associated with'*, *'correlates with'* are used throughout.
> No causal inference is claimed from cross-sectional OLS.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Paths ────────────────────────────────────────────────────────────────────
ROOT       = Path('..').resolve()
FEAT_PATH  = ROOT / 'data' / 'processed' / 'features_2022_judet.parquet'
GEO_PATH   = ROOT / 'app' / 'src' / 'data' / 'geojson' / 'ro_judete.geojson'
FIG_DIR    = ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)

SOURCE_NOTE = 'Source: Eurostat 2022, INS Census 2021, roaep.ro 2024/2025'

# ── Availability flags ───────────────────────────────────────────────────────
DATA_AVAILABLE = FEAT_PATH.exists()
GEO_AVAILABLE  = GEO_PATH.exists()

print(f"Feature matrix : {'OK' if DATA_AVAILABLE else 'NOT FOUND — run pipeline first'}")
print(f"GeoJSON        : {'OK' if GEO_AVAILABLE  else 'NOT FOUND — run download_eurostat.py'}")

# ── Plotting defaults ────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'figure.facecolor': 'white',
})
PARTY_COLORS = {
    't_AUR':  '#C0392B', 't_PSD': '#E74C3C', 't_PNL': '#F39C12',
    't_USR':  '#2980B9', 't_UDMR': '#27AE60', 't_SOS': '#8E44AD',
    't_POT':  '#16A085', 't_turnout': '#2C3E50',
}

## 1. Data Load & Completeness Audit

In [ ]:
if not DATA_AVAILABLE:
    print("Feature matrix not found. Download data and run the pipeline:")
    print("  python scripts/01_ingest/download_eurostat.py")
    print("  python scripts/01_ingest/run_all_ingest.py")
    print("  python scripts/02_clean/run_all_clean.py")
    print("  python scripts/03_features/build_features.py")
    raise SystemExit("Data not ready.")

df = pd.read_parquet(FEAT_PATH)
print(f"Shape: {df.shape}  ({len(df)} counties × {len(df.columns)} columns)")
print(f"\nColumn groups:")
for label, cols in [
    ('Identity',   [c for c in df.columns if c in ('siruta','judet','geo_nuts3','nuts2_code')]),
    ('Eurostat',   [c for c in df.columns if c in ('gdp_per_capita','unemployment_rate',
                    'poverty_risk_pct','tertiary_education_pct','early_school_leaving_pct',
                    'internet_use_pct','population_density','net_migration_rate',
                    'pct_pop_over65','pct_pop_under25','pct_employment_agriculture')]),
    ('Census',     [c for c in df.columns if c.startswith('pct_') and 'pop' not in c and 'employment' not in c]),
    ('Composites', [c for c in df.columns if c.startswith('idx_')]),
    ('Sentiment',  ['ep_2024_anti_estab','swing_anti_estab_2020_2024','georgescu_r1_2024','simion_r1_2025']),
    ('Targets',    [c for c in df.columns if c.startswith('t_')]),
    ('Validation', [c for c in df.columns if c.startswith('pres')]),
]:
    present = [c for c in cols if c in df.columns]
    print(f"  {label:12s}: {len(present)} columns")

In [ ]:
# Missing value heatmap
analysis_cols = [c for c in df.columns if c not in ('siruta','judet','geo_nuts3','nuts2_code')
                 and not c.startswith('source_nuts')]
missing = df[analysis_cols].isna()

fig, ax = plt.subplots(figsize=(14, max(5, len(analysis_cols) * 0.22)))
sns.heatmap(missing.T, cmap='RdYlGn_r', cbar=False, ax=ax,
            xticklabels=df['judet'].tolist(),
            yticklabels=analysis_cols, linewidths=0.3)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=6)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=7)
ax.set_title('Missing Value Map — dark = missing\n'
             '(green = present, red = missing)', fontsize=11)
fig.text(0.5, -0.01, SOURCE_NOTE, ha='center', fontsize=7, style='italic')
plt.tight_layout()
fig.savefig(FIG_DIR / 'eda_missing_heatmap.png', bbox_inches='tight')
plt.show()

# Summary table
miss_pct = missing.mean().mul(100).round(1)
miss_pct = miss_pct[miss_pct > 0].sort_values(ascending=False)
if miss_pct.empty:
    print("No missing values — complete feature matrix.")
else:
    print("Columns with missing values:")
    print(miss_pct.to_frame('% missing').to_string())

## 2. Electoral Results Overview — 2024 Parliamentary

In [ ]:
party_cols = [c for c in ['t_PSD','t_PNL','t_USR','t_AUR','t_UDMR','t_SOS','t_POT']
              if c in df.columns]

if not party_cols:
    print("No party share columns found. Check that election data was ingested.")
else:
    party_df = df[['judet'] + party_cols].set_index('judet').sort_index()

    # National averages
    nat_avg = party_df.mean() * 100
    print("National average vote shares (2024 Parliamentary):")
    for p, v in nat_avg.sort_values(ascending=False).items():
        bar = '█' * int(v / 2)
        print(f"  {p:10s}: {v:5.1f}%  {bar}")

    # Stacked bar chart
    colors = [PARTY_COLORS.get(c, '#95A5A6') for c in party_cols]
    fig, ax = plt.subplots(figsize=(16, 5))
    (party_df * 100).plot(kind='bar', stacked=True, ax=ax, color=colors,
                           width=0.85, edgecolor='white', linewidth=0.3)
    ax.set_xlabel('')
    ax.set_ylabel('Vote share (%)')
    ax.set_title('2024 Parliamentary Election — Vote Shares by County (Camera Deputaților)',
                 fontsize=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=7)
    ax.legend(loc='upper right', fontsize=8,
              labels=[c.replace('t_','') for c in party_cols])
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%d%%'))
    fig.text(0.5, -0.02, SOURCE_NOTE, ha='center', fontsize=7, style='italic')
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'eda_vote_shares_stacked.png', bbox_inches='tight')
    plt.show()

## 3. Geographic Distribution — Choropleth Maps

Maps require the GeoJSON (`app/src/data/geojson/ro_judete.geojson`).  
If unavailable, sorted bar charts are shown instead.

In [ ]:
MAP_TARGETS = [
    ('t_AUR',    'AUR vote share',     'Reds'),
    ('t_PSD',    'PSD vote share',     'YlOrRd'),
    ('t_USR',    'USR vote share',     'Blues'),
    ('t_UDMR',   'UDMR vote share',    'Greens'),
    ('t_turnout','Turnout',            'viridis'),
    ('idx_grievance', 'Grievance index','RdYlGn_r'),
]
MAP_TARGETS = [(col, label, cmap) for col, label, cmap in MAP_TARGETS if col in df.columns]

def _fallback_bar(col: str, label: str, cmap: str, ax: plt.Axes) -> None:
    """Sorted horizontal bar chart when GeoJSON is unavailable."""
    sub = df[['judet', col]].dropna().sort_values(col)
    vals = sub[col].values
    norm = plt.Normalize(vals.min(), vals.max())
    colors = plt.get_cmap(cmap)(norm(vals))
    ax.barh(sub['judet'], vals * 100 if vals.max() < 2 else vals,
            color=colors)
    ax.set_xlabel('%' if vals.max() < 2 else '')
    ax.set_title(label, fontsize=9)
    ax.tick_params(axis='y', labelsize=6)

if GEO_AVAILABLE:
    import geopandas as gpd
    gdf = gpd.read_file(GEO_PATH)
    # Join on NUTS3 code
    gdf = gdf.merge(df.rename(columns={'geo_nuts3': 'NUTS_ID'}),
                    on='NUTS_ID', how='left')
    n = len(MAP_TARGETS)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.ravel()
    for i, (col, label, cmap) in enumerate(MAP_TARGETS):
        if col not in gdf.columns:
            axes[i].set_visible(False)
            continue
        gdf.plot(column=col, ax=axes[i], cmap=cmap, legend=True,
                 legend_kwds={'shrink': 0.6, 'label': ''},
                 missing_kwds={'color': 'lightgrey'},
                 edgecolor='white', linewidth=0.3)
        axes[i].set_title(label, fontsize=10)
        axes[i].axis('off')
    for j in range(len(MAP_TARGETS), len(axes)):
        axes[j].set_visible(False)
    fig.suptitle('Geographic Distribution — Key Variables (2024)', fontsize=13, y=1.01)
    fig.text(0.5, -0.01, SOURCE_NOTE, ha='center', fontsize=7, style='italic')
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'eda_choropleth_maps.png', bbox_inches='tight')
    plt.show()
else:
    print("GeoJSON not available — showing sorted bar charts instead.")
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.ravel()
    for i, (col, label, cmap) in enumerate(MAP_TARGETS):
        _fallback_bar(col, label, cmap, axes[i])
    for j in range(len(MAP_TARGETS), len(axes)):
        axes[j].set_visible(False)
    fig.suptitle('County Rankings — Key Variables (GeoJSON unavailable)', fontsize=12)
    fig.text(0.5, -0.01, SOURCE_NOTE, ha='center', fontsize=7, style='italic')
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'eda_county_rankings.png', bbox_inches='tight')
    plt.show()

## 4. Feature Distributions

In [ ]:
euro_feats = [c for c in [
    'gdp_per_capita','unemployment_rate','poverty_risk_pct',
    'tertiary_education_pct','early_school_leaving_pct','internet_use_pct',
    'population_density','net_migration_rate','pct_pop_over65','pct_pop_under25',
    'pct_employment_agriculture'
] if c in df.columns]

census_feats = [c for c in [
    'pct_maghiari','pct_romi','pct_pentecostal','pct_reformed',
    'pct_greco_catholic','pct_urban'
] if c in df.columns]

all_feats = euro_feats + census_feats
ncols = 4
nrows = int(np.ceil(len(all_feats) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 2.8))
axes = axes.ravel()

for i, col in enumerate(all_feats):
    data = df[col].dropna()
    axes[i].hist(data, bins=12, color='#3498db', edgecolor='white', alpha=0.85)
    axes[i].axvline(data.mean(),  color='red',    lw=1.5, ls='--', label=f'Mean {data.mean():.2f}')
    axes[i].axvline(data.median(), color='orange', lw=1.5, ls=':',  label=f'Med  {data.median():.2f}')
    axes[i].set_title(col.replace('_', ' '), fontsize=8)
    axes[i].tick_params(labelsize=7)
    axes[i].legend(fontsize=6)
    sk = stats.skew(data)
    axes[i].text(0.97, 0.95, f'skew={sk:.2f}', transform=axes[i].transAxes,
                 ha='right', va='top', fontsize=6, color='grey')

for j in range(len(all_feats), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Distributions — Eurostat 2022 + INS Census 2021', fontsize=12)
fig.text(0.5, -0.01, SOURCE_NOTE, ha='center', fontsize=7, style='italic')
plt.tight_layout()
fig.savefig(FIG_DIR / 'eda_feature_distributions.png', bbox_inches='tight')
plt.show()

# Descriptive stats table
pd.set_option('display.float_format', '{:.3f}'.format)
df[all_feats].describe().T[['mean','std','min','50%','max']].rename(columns={'50%':'median'})

## 5. Bivariate Analysis — AUR Share vs Key Features

Testing H1 (grievance), H2 (education), H3 (depopulation), H4 (Pentecostal).

In [ ]:
if 't_AUR' not in df.columns:
    print("t_AUR not available.")
else:
    scatter_pairs = [
        ('idx_grievance',          'Economic Grievance Index',       'H1: Grievance → AUR'),
        ('tertiary_education_pct', 'Tertiary Education % (25–64)',   'H2: Education → AUR (neg.)'),
        ('net_migration_rate',     'Net Migration Rate',             'H3: Depopulation → AUR'),
        ('pct_pentecostal',        'Pentecostal %',                  'H4: Pentecostal → AUR'),
        ('unemployment_rate',      'Unemployment Rate',              'H1a: Unemployment → AUR'),
        ('pct_urban',              'Urban Population %',             'H6: Urban → AUR (neg.)'),
    ]
    scatter_pairs = [(x, xl, title) for x, xl, title in scatter_pairs if x in df.columns]

    ncols = 3
    nrows = int(np.ceil(len(scatter_pairs) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 4))
    axes = axes.ravel()

    for i, (x_col, x_label, title) in enumerate(scatter_pairs):
        sub = df[['judet', x_col, 't_AUR']].dropna()
        x, y = sub[x_col].values, sub['t_AUR'].values
        r, p = stats.pearsonr(x, y)
        slope, intercept, *_ = stats.linregress(x, y)

        axes[i].scatter(x, y * 100, alpha=0.7, edgecolors='black', lw=0.4,
                        s=50, color=PARTY_COLORS.get('t_AUR', '#e74c3c'), zorder=3)

        # Label outliers (top/bottom 15% by residual)
        residuals = y - (slope * x + intercept)
        thr = np.percentile(np.abs(residuals), 85)
        for _, row in sub.iterrows():
            res = row['t_AUR'] - (slope * row[x_col] + intercept)
            if abs(res) > thr:
                axes[i].annotate(row['judet'], (row[x_col], row['t_AUR'] * 100),
                                 fontsize=6, ha='left', va='bottom')

        x_line = np.linspace(x.min(), x.max(), 100)
        axes[i].plot(x_line, (slope * x_line + intercept) * 100,
                     'r-', lw=1.5, label=f'r={r:.3f}, p={p:.3f}')
        axes[i].set_xlabel(x_label, fontsize=9)
        axes[i].set_ylabel('AUR share (%)', fontsize=9)
        axes[i].set_title(title, fontsize=10)
        axes[i].legend(fontsize=8)
        axes[i].yaxis.set_major_formatter(mticker.PercentFormatter())

    for j in range(len(scatter_pairs), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle('AUR Vote Share vs Key Features — Bivariate Analysis', fontsize=12)
    fig.text(0.5, -0.01, SOURCE_NOTE, ha='center', fontsize=7, style='italic')
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'eda_aur_bivariate.png', bbox_inches='tight')
    plt.show()

## 6. Correlation Matrix — Feature Space Structure

In [ ]:
target_cols = [c for c in ['t_AUR','t_PSD','t_PNL','t_USR','t_UDMR','t_turnout'] if c in df.columns]
feat_cols   = [c for c in (
    euro_feats + census_feats +
    ['idx_grievance','idx_modernity','idx_demographic_pressure',
     'ep_2024_anti_estab','swing_anti_estab_2020_2024']
) if c in df.columns]

corr_cols = feat_cols + target_cols
corr_df   = df[corr_cols].corr()

# Shorten labels for readability
short = {
    'gdp_per_capita': 'GDP/cap', 'unemployment_rate': 'Unempl',
    'poverty_risk_pct': 'Poverty', 'tertiary_education_pct': 'TertEd',
    'early_school_leaving_pct': 'ESL', 'internet_use_pct': 'Internet',
    'population_density': 'PopDens', 'net_migration_rate': 'NetMig',
    'pct_pop_over65': '%65+', 'pct_pop_under25': '%U25',
    'pct_employment_agriculture': '%AgriEmp',
    'pct_maghiari': '%Magyar', 'pct_romi': '%Roma',
    'pct_pentecostal': '%Pent', 'pct_reformed': '%Reform',
    'pct_greco_catholic': '%GrecoCath', 'pct_urban': '%Urban',
    'idx_grievance': 'Grievance', 'idx_modernity': 'Modernity',
    'idx_demographic_pressure': 'DemPress',
    'ep_2024_anti_estab': 'EP24_anti', 'swing_anti_estab_2020_2024': 'Swing20_24',
    't_AUR': 'AUR', 't_PSD': 'PSD', 't_PNL': 'PNL',
    't_USR': 'USR', 't_UDMR': 'UDMR', 't_turnout': 'Turnout',
}
corr_df = corr_df.rename(index=short, columns=short)

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(corr_df, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            annot_kws={'size': 6}, linewidths=0.3,
            cbar_kws={'shrink': 0.6})
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=7)

# Highlight target block
n_feat = len(feat_cols)
ax.add_patch(plt.Rectangle((n_feat, 0), len(target_cols), n_feat,
                             fill=False, edgecolor='black', lw=2))
ax.set_title('Pearson Correlation Matrix — Features × Targets\n'
             '(black box = feature–target correlations)', fontsize=12)
fig.text(0.5, -0.01, SOURCE_NOTE, ha='center', fontsize=7, style='italic')
plt.tight_layout()
fig.savefig(FIG_DIR / 'eda_correlation_matrix.png', bbox_inches='tight')
plt.show()

# Top-5 correlates of AUR
if 'AUR' in corr_df.columns:
    print("Top correlates with AUR share:")
    print(corr_df['AUR'].drop('AUR').abs().sort_values(ascending=False).head(10).to_string())

## 7. PCA — Feature Space Structure

PCA reveals multicollinearity structure and identifies the main axes of variation
across counties. Counties far from the origin are structural outliers.  
Expected: PC1 ≈ economic development axis; PC2 ≈ ethnic/geographic axis.

In [ ]:
pca_cols = [c for c in feat_cols if c in df.columns]
pca_df   = df[['judet'] + pca_cols + target_cols].dropna(subset=pca_cols)
X_raw    = pca_df[pca_cols].values

scaler = StandardScaler()
X_std  = scaler.fit_transform(X_raw)

pca = PCA(n_components=min(len(pca_cols), len(pca_df)))
Z   = pca.fit_transform(X_std)

explained = pca.explained_variance_ratio_ * 100
print("Explained variance by component:")
cumsum = 0
for i, v in enumerate(explained[:8]):
    cumsum += v
    print(f"  PC{i+1}: {v:.1f}%  (cumulative: {cumsum:.1f}%)")

# Biplot — PC1 vs PC2, coloured by AUR share
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: county scatter coloured by AUR share
if 't_AUR' in pca_df.columns:
    aur_vals = pca_df['t_AUR'].fillna(0).values
    sc = axes[0].scatter(Z[:, 0], Z[:, 1], c=aur_vals, cmap='Reds',
                         s=60, edgecolors='black', lw=0.4, zorder=3)
    plt.colorbar(sc, ax=axes[0], label='AUR vote share')

for i, (xi, yi, name) in enumerate(zip(Z[:, 0], Z[:, 1], pca_df['judet'])):
    if abs(xi) > np.percentile(np.abs(Z[:, 0]), 80) or abs(yi) > np.percentile(np.abs(Z[:, 1]), 80):
        axes[0].annotate(name, (xi, yi), fontsize=6, ha='left', va='bottom')

axes[0].axhline(0, color='grey', lw=0.5)
axes[0].axvline(0, color='grey', lw=0.5)
axes[0].set_xlabel(f'PC1 ({explained[0]:.1f}%)')
axes[0].set_ylabel(f'PC2 ({explained[1]:.1f}%)')
axes[0].set_title('County Scores — PC1 vs PC2\n(coloured by AUR share)')

# Right: loadings biplot (top 10 features by PC1 + PC2 magnitude)
loadings = pd.DataFrame(pca.components_[:2].T, index=pca_cols, columns=['PC1','PC2'])
loadings['magnitude'] = np.sqrt(loadings['PC1']**2 + loadings['PC2']**2)
top_load = loadings.nlargest(10, 'magnitude')
short_labels = {c: short.get(c, c) for c in pca_cols}

for feat, row in top_load.iterrows():
    axes[1].arrow(0, 0, row['PC1'], row['PC2'],
                  head_width=0.02, head_length=0.02,
                  fc='#2980B9', ec='#2980B9', alpha=0.7)
    axes[1].text(row['PC1'] * 1.1, row['PC2'] * 1.1,
                 short_labels.get(feat, feat), fontsize=8, ha='center')
axes[1].axhline(0, color='grey', lw=0.5)
axes[1].axvline(0, color='grey', lw=0.5)
axes[1].set_xlabel(f'PC1 ({explained[0]:.1f}%)')
axes[1].set_ylabel(f'PC2 ({explained[1]:.1f}%)')
axes[1].set_title('Feature Loadings — Top 10 by Magnitude')
circle = plt.Circle((0, 0), 1, fill=False, color='grey', lw=0.5, ls='--')
axes[1].add_patch(circle)
axes[1].set_xlim(-1.3, 1.3)
axes[1].set_ylim(-1.3, 1.3)
axes[1].set_aspect('equal')

fig.suptitle('PCA Biplot — Feature Space Structure', fontsize=12)
fig.text(0.5, -0.01, SOURCE_NOTE, ha='center', fontsize=7, style='italic')
plt.tight_layout()
fig.savefig(FIG_DIR / 'eda_pca_biplot.png', bbox_inches='tight')
plt.show()

## 8. Ethnic Bloc Validation — UDMR vs Hungarian %

**Hypothesis H4a:** Hungarian ethnicity % is a near-linear predictor of UDMR share.  
This is a sanity check — if the relationship is not strong (r > 0.95), there is a data quality issue.

In [ ]:
if 'pct_maghiari' not in df.columns or 't_UDMR' not in df.columns:
    print("pct_maghiari or t_UDMR not available.")
else:
    sub = df[['judet','pct_maghiari','t_UDMR']].dropna()
    x, y = sub['pct_maghiari'].values, sub['t_UDMR'].values
    r, p = stats.pearsonr(x, y)
    slope, intercept, *_ = stats.linregress(x, y)

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(x * 100, y * 100, edgecolors='black', lw=0.5, s=60,
               color='#27AE60', alpha=0.8, zorder=3)
    for _, row in sub.iterrows():
        if row['pct_maghiari'] > 0.05 or row['t_UDMR'] > 0.05:
            ax.annotate(row['judet'], (row['pct_maghiari'] * 100, row['t_UDMR'] * 100),
                        fontsize=7)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line * 100, (slope * x_line + intercept) * 100,
            'r-', lw=2, label=f'r={r:.3f}, p={p:.4f}')
    ax.set_xlabel('Hungarian ethnicity % (Census 2021)')
    ax.set_ylabel('UDMR vote share % (Parliamentary 2024)')
    ax.set_title('Ethnic Bloc Validation: UDMR Share vs Hungarian Ethnicity %\n'
                 '(should be near-linear, r > 0.95)')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    fig.text(0.5, -0.01, SOURCE_NOTE, ha='center', fontsize=7, style='italic')
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'eda_udmr_hungarian.png', bbox_inches='tight')
    plt.show()

    quality = 'PASS' if r > 0.93 else 'WARN — check data'
    print(f"Ethnic bloc validation: r={r:.3f}  [{quality}]")
    print("Note: pct_maghiari will be a forced (non-penalised) control in all models.")

## 9. Pentecostal % as Anti-Establishment Proxy

**Hypothesis H4b:** Pentecostal % predicts AUR/Georgescu share independently of economic deprivation.  
Concentration expected in NW Romania (Bihor, Satu Mare) and parts of NE (Suceava, Bacău).

In [ ]:
pent_targets = [(c, c.replace('t_','').replace('georgescu_r1_2024','Georgescu R1 2024'))
                for c in ['t_AUR','georgescu_r1_2024'] if c in df.columns]

if 'pct_pentecostal' not in df.columns or not pent_targets:
    print("pct_pentecostal or AUR/Georgescu columns not available.")
else:
    fig, axes = plt.subplots(1, len(pent_targets), figsize=(6 * len(pent_targets), 5))
    if len(pent_targets) == 1:
        axes = [axes]

    for ax, (target_col, target_label) in zip(axes, pent_targets):
        sub = df[['judet','pct_pentecostal', target_col]].dropna()
        x, y = sub['pct_pentecostal'].values, sub[target_col].values
        r, p_val = stats.pearsonr(x, y)
        slope, intercept, *_ = stats.linregress(x, y)

        ax.scatter(x * 100, y * 100, edgecolors='black', lw=0.4, s=55,
                   color='#8E44AD', alpha=0.8, zorder=3)
        for _, row in sub.iterrows():
            if row['pct_pentecostal'] > np.percentile(x, 70):
                ax.annotate(row['judet'], (row['pct_pentecostal'] * 100, row[target_col] * 100),
                            fontsize=6, ha='left', va='bottom')
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line * 100, (slope * x_line + intercept) * 100,
                'r-', lw=1.5, label=f'r={r:.3f}, p={p_val:.3f}')
        ax.set_xlabel('Pentecostal % (Census 2021)')
        ax.set_ylabel(f'{target_label} share (%)')
        ax.set_title(f'Pentecostal % vs {target_label}')
        ax.legend(fontsize=8)
        ax.xaxis.set_major_formatter(mticker.PercentFormatter())
        ax.yaxis.set_major_formatter(mticker.PercentFormatter())

    fig.suptitle('H4b: Pentecostal % as Religious Conservative Proxy', fontsize=11)
    fig.text(0.5, -0.01, SOURCE_NOTE, ha='center', fontsize=7, style='italic')
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'eda_pentecostal_hypothesis.png', bbox_inches='tight')
    plt.show()

## 10. Georgescu → Simion Vote Transfer

**Hypothesis H7:** The nationalist vote was geographically stable after the annulment.  
Expected: r > 0.80. Counties far from the regression line are where the vote redistributed most.

In [ ]:
if 'georgescu_r1_2024' not in df.columns or 'simion_r1_2025' not in df.columns:
    print("georgescu_r1_2024 or simion_r1_2025 not available — download presidential R1 data.")
else:
    sub = df[['judet','georgescu_r1_2024','simion_r1_2025']].dropna()
    x, y = sub['georgescu_r1_2024'].values, sub['simion_r1_2025'].values
    r, p_val  = stats.pearsonr(x, y)
    slope, intercept, *_ = stats.linregress(x, y)

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.scatter(x * 100, y * 100, s=65, edgecolors='black', lw=0.5,
               color='#C0392B', alpha=0.8, zorder=3)
    for _, row in sub.iterrows():
        ax.annotate(row['judet'],
                    (row['georgescu_r1_2024'] * 100, row['simion_r1_2025'] * 100),
                    fontsize=6, ha='left', va='bottom', alpha=0.75)

    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line * 100, (slope * x_line + intercept) * 100,
            'r-', lw=2, label=f'Linear fit  r={r:.3f}, p={p_val:.4f}')
    # 1:1 line
    lim_max = max(x.max(), y.max()) * 100 * 1.05
    ax.plot([0, lim_max], [0, lim_max], 'k--', lw=0.8, alpha=0.4,
            label='1:1 (perfect transfer)')

    ax.set_xlabel('Georgescu vote share — Presidential R1, Nov 2024 (ANNULLED)')
    ax.set_ylabel('Simion vote share — Presidential R1, May 2025 (redo election)')
    ax.set_title('Georgescu → Simion Vote Transfer\n'
                 'Tests geographic stability of the nationalist vote after annulment')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    fig.text(0.5, -0.01, SOURCE_NOTE, ha='center', fontsize=7, style='italic')
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'eda_georgescu_simion_transfer.png', bbox_inches='tight')
    plt.show()

    transfer_quality = 'geographically STABLE' if r > 0.80 else 'REDISTRIBUTED after annulment'
    print(f"Transfer correlation: r={r:.3f}  → Nationalist vote was {transfer_quality}")

    # Biggest deviations
    sub = sub.copy()
    sub['predicted_simion'] = slope * sub['georgescu_r1_2024'] + intercept
    sub['transfer_residual'] = sub['simion_r1_2025'] - sub['predicted_simion']
    print("\nCounties where Simion EXCEEDED Georgescu prediction (Simion over-performed):")
    print(sub.nlargest(5, 'transfer_residual')[['judet','georgescu_r1_2024','simion_r1_2025','transfer_residual']].to_string(index=False))
    print("\nCounties where Simion FELL SHORT of Georgescu prediction (vote loss):")
    print(sub.nsmallest(5, 'transfer_residual')[['judet','georgescu_r1_2024','simion_r1_2025','transfer_residual']].to_string(index=False))

## 11. Urban–Rural Cleavage — R2 2025 (Dan vs Simion)

In [ ]:
if 'pct_urban' not in df.columns or 'pres25r2_DAN' not in df.columns:
    print("pct_urban or pres25r2_DAN not available.")
else:
    sub = df[['judet','pct_urban','pres25r2_DAN','pres25r2_SIMION']].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, (col, label, color) in zip(axes, [
        ('pres25r2_DAN',   'Dan R2 2025',   '#2980B9'),
        ('pres25r2_SIMION','Simion R2 2025','#C0392B'),
    ]):
        x, y = sub['pct_urban'].values, sub[col].values
        r, p_val = stats.pearsonr(x, y)
        slope, intercept, *_ = stats.linregress(x, y)
        ax.scatter(x * 100, y * 100, s=55, edgecolors='black', lw=0.4,
                   color=color, alpha=0.8, zorder=3)
        residuals = y - (slope * x + intercept)
        thr = np.percentile(np.abs(residuals), 80)
        for _, row in sub.iterrows():
            res = row[col] - (slope * row['pct_urban'] + intercept)
            if abs(res) > thr:
                ax.annotate(row['judet'], (row['pct_urban'] * 100, row[col] * 100),
                            fontsize=6, ha='left', va='bottom')
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line * 100, (slope * x_line + intercept) * 100,
                'k-', lw=1.5, label=f'r={r:.3f}, p={p_val:.3f}')
        ax.set_xlabel('Urban population % (Census 2021)')
        ax.set_ylabel(f'{label} share (%)')
        ax.set_title(f'Urban % vs {label}')
        ax.legend(fontsize=8)
        ax.xaxis.set_major_formatter(mticker.PercentFormatter())
        ax.yaxis.set_major_formatter(mticker.PercentFormatter())

    fig.suptitle('H6: Urban–Rural Cleavage — Presidential R2 2025', fontsize=11)
    fig.text(0.5, -0.01, SOURCE_NOTE, ha='center', fontsize=7, style='italic')
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'eda_urban_r2_cleavage.png', bbox_inches='tight')
    plt.show()

## 12. Swing Analysis — Anti-Establishment 2020 → 2024

In [ ]:
if 'swing_anti_estab_2020_2024' not in df.columns:
    print("Swing column not available — requires both 2020 and 2024 parliamentary data.")
else:
    sub = df[['judet','swing_anti_estab_2020_2024','nuts2_code']].dropna().copy()
    sub = sub.sort_values('swing_anti_estab_2020_2024')
    colors = ['#E74C3C' if v > 0 else '#3498DB' for v in sub['swing_anti_estab_2020_2024']]

    fig, ax = plt.subplots(figsize=(9, max(7, len(sub) * 0.3)))
    bars = ax.barh(sub['judet'], sub['swing_anti_estab_2020_2024'] * 100,
                   color=colors, edgecolor='white', linewidth=0.3)
    ax.axvline(0, color='black', lw=1)

    # Annotate NUTS 2 region
    if sub['nuts2_code'].notna().any():
        nuts2_colors = {'RO11':'#3498DB','RO12':'#2ECC71','RO21':'#E74C3C',
                        'RO22':'#F39C12','RO31':'#9B59B6','RO32':'#1ABC9C',
                        'RO41':'#E67E22','RO42':'#34495E'}
        for i, (_, row) in enumerate(sub.iterrows()):
            n2 = row['nuts2_code']
            if pd.notna(n2) and n2 in nuts2_colors:
                ax.get_yticklabels()[i].set_color(nuts2_colors[n2])

    ax.set_xlabel('Change in anti-establishment vote share (pp), 2020→2024 Parliamentary')
    ax.set_title('Anti-Establishment Swing by County — Parliamentary 2020→2024\n'
                 '(AUR + SOS + POT combined; yticklabel colour = NUTS 2 region)')
    ax.tick_params(axis='y', labelsize=7)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0fpp'))
    fig.text(0.5, -0.01, SOURCE_NOTE, ha='center', fontsize=7, style='italic')
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'eda_swing_2020_2024.png', bbox_inches='tight')
    plt.show()

    print(f"Mean swing: {sub['swing_anti_estab_2020_2024'].mean()*100:+.1f}pp")
    print(f"Largest swing UP:   {sub.iloc[-1]['judet']:20s} "
          f"+{sub.iloc[-1]['swing_anti_estab_2020_2024']*100:.1f}pp")
    print(f"Largest swing DOWN: {sub.iloc[0]['judet']:20s}  "
          f"{sub.iloc[0]['swing_anti_estab_2020_2024']*100:.1f}pp")

## 13. Pre-Modelling Notes — VIF and Outliers

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

model_feat_cols = [c for c in [
    'gdp_per_capita','unemployment_rate','poverty_risk_pct',
    'tertiary_education_pct','early_school_leaving_pct','internet_use_pct',
    'population_density','net_migration_rate','pct_pop_over65','pct_pop_under25',
    'pct_employment_agriculture',
    'pct_maghiari','pct_romi','pct_orthodox','pct_pentecostal','pct_reformed','pct_urban',
    'idx_grievance','idx_modernity','idx_demographic_pressure',
] if c in df.columns]

X_vif = df[model_feat_cols].dropna()
scaler_vif = StandardScaler()
X_vif_scaled = pd.DataFrame(scaler_vif.fit_transform(X_vif),
                             columns=model_feat_cols)

vif_df = pd.DataFrame({
    'Feature': model_feat_cols,
    'VIF': [variance_inflation_factor(X_vif_scaled.values, i)
            for i in range(len(model_feat_cols))]
}).sort_values('VIF', ascending=False)

vif_df['Flag'] = vif_df['VIF'].apply(
    lambda v: '⚠ HIGH — use composite' if v > 10
              else ('moderate' if v > 5 else 'OK')
)
print("VIF Table (N={} counties):".format(len(X_vif)))
print(vif_df.to_string(index=False))
print("\nNote: Composite indices (idx_grievance etc.) were designed to address high-VIF clusters.")
print("In Elastic Net models, use EITHER raw features OR composite indices, not both.")

In [ ]:
# Leverage / outlier identification for AUR model
import statsmodels.api as sm

if 't_AUR' in df.columns:
    # Use composite indices + ethnic variables to avoid multicollinearity in this preview
    preview_cols = [c for c in [
        'idx_grievance','idx_modernity','idx_demographic_pressure',
        'pct_maghiari','pct_pentecostal','pct_urban'
    ] if c in df.columns]

    sub = df[['judet'] + preview_cols + ['t_AUR']].dropna()
    X_p = sm.add_constant(StandardScaler().fit_transform(sub[preview_cols]))
    y_p = sub['t_AUR'].values

    ols_res  = sm.OLS(y_p, X_p).fit()
    influence = ols_res.get_influence()
    cooks_d   = influence.cooks_distance[0]

    cooks_df = pd.DataFrame({
        'judet': sub['judet'].values,
        'cooks_d': cooks_d,
        't_AUR': sub['t_AUR'].values,
    }).sort_values('cooks_d', ascending=False)

    threshold = 4 / len(sub)
    print(f"Cook's Distance threshold (4/N={len(sub)}): {threshold:.4f}")
    print("\nHigh-leverage counties (Cook's D > threshold):")
    print(cooks_df[cooks_df['cooks_d'] > threshold].to_string(index=False))
    print("\nNote: București-Ilfov (RO32) is typically a structural outlier — "
          "run models with and without it.")

## Summary & Next Steps

### What to look for in the outputs above

| Section | Key check | Expected finding |
|---|---|---|
| Completeness | No red in missing map | All 42 counties complete |
| UDMR validation | r > 0.93 | Near-linear ethnic bloc |
| Georgescu→Simion | r > 0.80 | Geographically stable nationalist vote |
| PCA PC1 | Education + GDP vs Poverty loading | Development axis |
| VIF table | No raw features > 10 if composites used | Composites resolve collinearity |
| Cook's D | Budapest-Ilfov flagged | Structural outlier confirmed |

### Proceed to modelling once:
- [ ] Feature matrix is complete (no missing target columns)
- [ ] UDMR validation passes (r > 0.93)
- [ ] Georgescu→Simion transfer is assessed
- [ ] VIF structure understood — choose raw vs composite features per model

```python
# Run models:
# python scripts/04_model/run_all_models.py
```